In [ ]:
import os
import sys
module_path = os.path.abspath(os.path.join('../module/'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
from prostate_recurrence.image_processing import tissue_mask

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from typing import Dict, Tuple
from PIL import Image

In [ ]:
data_path = "/Volumes/proj-sahai-tme-ml/working/raw_data/clinical/prostate/chiip_cohort/PSR_20X/C023036_B04_PSR_20X.czi"

In [ ]:
save_path = "/Volumes/proj-sahai-tme-ml/working/processed_data/pre_processed_data/clinical/prostate/chiip_cohort/slide_20X/C023036_B04/tile_size_2000/whole_slide/PSR/masks/"
os.makedirs(save_path, exist_ok=True)
os.chmod(save_path, mode=0o777)

In [ ]:
dict_imgs = tissue_mask.read_image(
    path_to_img=data_path,
)

In [ ]:
dict_masks = tissue_mask.create_tissue_mask(
    dict_imgs=dict_imgs,
    gaussian_blur_sigma=2.,
    resize_factor=4,
    erosion_n=5,
    dilation_n=50
)

In [ ]:
dict_masks.keys()

In [ ]:
plt.imshow(dict_imgs['ScanRegion0'])

In [ ]:
plt.imshow(dict_masks['ScanRegion0'], cmap=plt.cm.Greys)

In [ ]:
dict_mask_tiles_all_scenes = {}
for scene, mask in dict_masks.items():
    mask_uint8 = (mask*255).astype(np.uint8)
    (dict_mask_tiles, nrow, ncol) = tissue_mask.create_mask_tiles(
        img=mask_uint8, 
        size=2000
    )
    dict_mask_tiles_all_scenes[scene] = (dict_mask_tiles, nrow, ncol)

In [ ]:
save_mask_tiles(
    dict_img_tiles=dict_mask_tiles,
    size=2000,
    nrow=nrow,
    ncol=ncol,
    save_path=save_path
)

In [ ]:

def save_mask_tiles(
    dict_img_tiles: Dict[int, np.ndarray],
    nrow: int,
    ncol: int,
    save_path: str,
    size: int = 512,
):
    zero_padding = 5
    for irow in range(nrow):
        if irow % 5 == 0:
            print(f"... saving {ncol} tiles in row {irow + 1} ...")
        for icol in range(ncol):
            tile_id = irow * ncol + icol
            img_tile = dict_img_tiles[tile_id]
            im = Image.fromarray(img_tile, mode="L")

            im.save(
                os.path.join(
                    save_path,
                    "image_tile_{}_{}_mask.tif".format(
                        str(irow).zfill(zero_padding), str(icol).zfill(zero_padding)
                    ),
                )
            )
